# 🔁 Tool Idempotency — The Bug That Emails Your Boss 5 Times

## Making Agent Tools Safe to Re-Run (with a REAL Gmail `send_email`)

| Detail | Info |
|---|---|
| **Topic** | Tool idempotency for AI agents |
| **Follows** | Session 04 (Tools) + Session 05 (Structured Outputs) |
| **Tool we build** | A real `send_email` that sends through the Gmail API |
| **Libraries** | `google-api-python-client`, `google-auth-oauthlib`, `python-dotenv` |
| **Credentials** | Google OAuth `client_secret.json` (loaded from a path in `.env`) |

### The one-line problem

Our agent is now **predictable** (structured output) — but it is not yet **safe**. An agent runs inside a **retry loop**, so the *same tool can fire more than once*. If that tool only **reads** data, no problem. If it **sends an email**, you just emailed the customer 5 times.

```
Agent: "Send refund confirmation"   → email sent ✅
Agent: *didn't get a clean reply, retries*
Agent: "Send refund confirmation"   → ANOTHER email 😱
Agent: *retries again*              → and ANOTHER 😱😱
```

> 💡 **Goal:** build a `send_email` tool, watch the broken version send duplicates, then add **one idempotency key** so it sends exactly once — no matter how many times the agent calls it.

---
## Part 0 — Setup & Credentials (safe handling)

We never hard-code the secret. Put these in your `.env` (already git-ignored):

```
GMAIL_CLIENT_SECRET=/absolute/path/to/client_secret.json
GMAIL_TOKEN=token.json
DEMO_RECIPIENT=your-own-email@gmail.com
```

> ⚠️ Keep `client_secret.json` **outside** any screen recording and outside git. `token.json` is created on first login and is also git-ignored.

In [ ]:
!pip3 install google-api-python-client google-auth-oauthlib python-dotenv -q

In [ ]:
import os
import base64
import hashlib
import json
from email.message import EmailMessage

from dotenv import load_dotenv

load_dotenv()

# Paths/values come from .env — nothing secret is written in this notebook.
CLIENT_SECRET_PATH = os.getenv("GMAIL_CLIENT_SECRET", "client_secret.json")
TOKEN_PATH = os.getenv("GMAIL_TOKEN", "token.json")
DEMO_RECIPIENT = os.getenv("DEMO_RECIPIENT", "your-own-email@gmail.com")

# Safety switch: True = simulate the send (no real email). False = REALLY send.
# Keep True while learning. Flip to False for the live 'it actually sent!' demo.
DRY_RUN = True

print("✅ Setup complete!")
print(f"   Recipient: {DEMO_RECIPIENT}")
print(f"   DRY_RUN  : {DRY_RUN}  (flip to False to send for real)")

---
## Part 1 — Build the REAL `send_email` tool

First, the part that actually talks to Gmail. This is the **side effect** — the line that changes the outside world.

`get_gmail_service()` logs you in once (browser popup) and caches the token. `_send_via_gmail()` does the real send and returns Gmail's message id.

In [ ]:
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/gmail.send"]


def get_gmail_service():
    """Authenticate once, reuse the cached token afterwards."""
    creds = None
    if os.path.exists(TOKEN_PATH):
        creds = Credentials.from_authorized_user_file(TOKEN_PATH, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRET_PATH, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_PATH, "w") as f:
            f.write(creds.to_json())
    return build("gmail", "v1", credentials=creds)


def _send_via_gmail(to: str, subject: str, body: str) -> str:
    """The actual side effect: send one email, return Gmail's message id."""
    if DRY_RUN:
        # Simulate a unique server-side id so the logic is identical to real sends.
        fake_id = "DRYRUN-" + hashlib.md5(os.urandom(8)).hexdigest()[:10]
        return fake_id
    service = get_gmail_service()
    msg = EmailMessage()
    msg.set_content(body)
    msg["To"] = to
    msg["Subject"] = subject
    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()
    sent = service.users().messages().send(userId="me", body={"raw": raw}).execute()
    return sent["id"]


print("✅ Real Gmail send wired up (DRY_RUN controls whether it actually fires)")

---
## Part 2 — The NON-idempotent tool (the danger, live)

Here is the tool the way most people write it first. It works perfectly... once.

We keep a `sent_log` so we can *count* how many emails actually went out.

In [ ]:
sent_log = []  # every real send appends here


def send_email_bad(to: str, subject: str, body: str) -> str:
    """Send an email. NOT idempotent — every call sends again."""
    message_id = _send_via_gmail(to, subject, body)
    sent_log.append(message_id)
    return json.dumps({"status": "sent", "message_id": message_id, "to": to})


# Simulate the agent's retry loop: it calls the SAME tool 3 times
sent_log.clear()
print("🤖 Agent decides to send the refund confirmation...")
print(send_email_bad(DEMO_RECIPIENT, "Your refund is confirmed", "Hi! Your refund of ₹2500 is approved."))
print("🤖 ...didn't get a clean reply, retries...")
print(send_email_bad(DEMO_RECIPIENT, "Your refund is confirmed", "Hi! Your refund of ₹2500 is approved."))
print("🤖 ...still unsure, retries again...")
print(send_email_bad(DEMO_RECIPIENT, "Your refund is confirmed", "Hi! Your refund of ₹2500 is approved."))

print(f"\n📨 Emails actually sent: {len(sent_log)}  ← should be 1, but it's {len(sent_log)}! 😱")

### What just happened?

Three calls → **three emails**. The customer now thinks they're getting three refunds.

The technical vocabulary for this:

- **Side effect** — the call changed the outside world (an email left the building).
- **Not idempotent** — doing it 3 times ≠ doing it once.
- **At-least-once delivery** — over a network you usually can't guarantee "exactly once"; the agent retries because it genuinely can't tell if the first one worked.

> 🧠 **Rule:** a tool that only *reads* (get_weather, get_stats) is naturally idempotent. A tool with a *side effect* (send_email, charge_card, place_order) is not — and must be designed to be safe.

---
## Part 3 — The fix: an idempotency key

The idea in one breath: **give every action a unique id, and remember the ids you've already done.**

```
first time you see key K  → do the side effect once, store the result under K
next time you see key K   → DON'T do it again, return the stored result
```

This is the same idea as a payment `idempotency key`, a `request_id`, or a `dedup key`.

In [ ]:
# Our 'I already did this' memory. In production this is Redis / a DB row.
processed = {}


def send_email_safe(to: str, subject: str, body: str, idempotency_key: str) -> str:
    """Send an email exactly once per idempotency_key. Safe to retry."""
    # Already handled this key? Return the SAME result, do not send again.
    if idempotency_key in processed:
        cached = processed[idempotency_key]
        return json.dumps({
            "status": "duplicate_ignored",
            "message_id": cached["message_id"],
            "to": cached["to"],
            "note": "already sent for this key — no new email",
        })

    # First time for this key → do the side effect once, then remember it.
    message_id = _send_via_gmail(to, subject, body)
    sent_log.append(message_id)
    processed[idempotency_key] = {"message_id": message_id, "to": to}
    return json.dumps({"status": "sent", "message_id": message_id, "to": to})


# Same retry loop — but now every call carries the SAME idempotency key
sent_log.clear()
processed.clear()
key = "refund-email-customer-42"

print("🤖 Agent sends the refund confirmation (key: refund-email-customer-42)...")
print(send_email_safe(DEMO_RECIPIENT, "Your refund is confirmed", "Hi! Your refund of ₹2500 is approved.", key))
print("🤖 ...retries...")
print(send_email_safe(DEMO_RECIPIENT, "Your refund is confirmed", "Hi! Your refund of ₹2500 is approved.", key))
print("🤖 ...retries again...")
print(send_email_safe(DEMO_RECIPIENT, "Your refund is confirmed", "Hi! Your refund of ₹2500 is approved.", key))

print(f"\n📨 Emails actually sent: {len(sent_log)}  ← correctly just {len(sent_log)}! ✅")

Three calls → **one email**. The agent still got a successful-looking response every time (so its loop is happy), but the *side effect* happened exactly once.

### Where does the key come from?

Two common strategies:

1. **Caller-provided id** — the agent/business logic supplies a meaningful key like `refund-email-customer-42`. Best when there's a natural "one per X" rule.
2. **Deterministic hash of the arguments** — when there's no natural id, hash the inputs so identical requests collapse to the same key automatically.

In [ ]:
def make_idempotency_key(to: str, subject: str, body: str) -> str:
    """Deterministic key: identical email content => identical key."""
    payload = f"{to}|{subject}|{body}".encode()
    return "auto-" + hashlib.sha256(payload).hexdigest()[:16]


k1 = make_idempotency_key(DEMO_RECIPIENT, "Your refund is confirmed", "Hi! Your refund of ₹2500 is approved.")
k2 = make_idempotency_key(DEMO_RECIPIENT, "Your refund is confirmed", "Hi! Your refund of ₹2500 is approved.")
k3 = make_idempotency_key(DEMO_RECIPIENT, "Your refund is confirmed", "Hi! Your refund of ₹5000 is approved.")

print(f"Same content  -> same key:      {k1 == k2}  ({k1})")
print(f"Different body -> different key: {k1 != k3}  ({k3})")

---
## Part 4 — Plug it into an agent tool

When you expose this to the LLM, make `idempotency_key` part of the tool schema so the agent passes it through on every retry.

In [ ]:
send_email_tool_schema = {
    "type": "function",
    "function": {
        "name": "send_email",
        "description": "Send an email. Idempotent: reuse the SAME idempotency_key when retrying so the email is sent only once.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string", "description": "Recipient email address"},
                "subject": {"type": "string"},
                "body": {"type": "string"},
                "idempotency_key": {
                    "type": "string",
                    "description": "Stable unique id for this send, e.g. 'refund-email-customer-42'. Reuse on retries.",
                },
            },
            "required": ["to", "subject", "body", "idempotency_key"],
        },
    },
}

available_tools = {"send_email": send_email_safe}

# Simulate the agent emitting the same tool call twice (e.g. after a timeout)
sent_log.clear()
processed.clear()
agent_tool_call = {
    "to": DEMO_RECIPIENT,
    "subject": "Welcome aboard!",
    "body": "Thanks for signing up.",
    "idempotency_key": "welcome-user-1001",
}

for attempt in range(2):
    print(f"call {attempt + 1}: {available_tools['send_email'](**agent_tool_call)}")

print(f"\n📨 Emails actually sent: {len(sent_log)}  ✅")

---
## Part 5 — A REAL agent calls the tool (Groq + Llama 3.3)

Now let's hand the tool to an actual LLM and let *it* decide to send the email. We use **Groq's `llama-3.3-70b-versatile`** through the standard **OpenAI client** (just point `base_url` at Groq).

The key teaching move: we **derive the idempotency key from the email content** inside our executor, so we don't even have to trust the model to provide a stable key. Run the same agent request twice (a replay / double-trigger) and only **one** email goes out.

In [ ]:
from openai import OpenAI

# Groq exposes an OpenAI-compatible API — same client, different base_url.
llm = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

# Tool schema the model sees. Note: NO idempotency_key here —
# we derive it ourselves from the content so the model can't get it wrong.
agent_tools = [
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email to a recipient.",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string", "description": "Recipient email address"},
                    "subject": {"type": "string"},
                    "body": {"type": "string"},
                },
                "required": ["to", "subject", "body"],
            },
        },
    }
]


def execute_send_email(to: str, subject: str, body: str) -> str:
    """Tool executor: derive a content-based key, then call the idempotent tool."""
    key = make_idempotency_key(to, subject, body)
    return send_email_safe(to, subject, body, idempotency_key=key)


def run_agent(user_request: str, max_steps: int = 4, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": "You send emails using the send_email tool. Call the tool to send the email, then briefly confirm to the user."},
        {"role": "user", "content": user_request},
    ]
    for step in range(max_steps):
        resp = llm.chat.completions.create(model=MODEL, messages=messages, tools=agent_tools)
        choice = resp.choices[0]
        if choice.finish_reason == "stop":
            return choice.message.content
        if choice.message.tool_calls:
            messages.append(choice.message)
            for tc in choice.message.tool_calls:
                args = json.loads(tc.function.arguments)
                if verbose:
                    print(f"  🔧 send_email(to={args.get('to')}, subject={args.get('subject')!r})")
                result = execute_send_email(**args)
                if verbose:
                    print(f"     → {result}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "(max steps reached)"


print("✅ Groq Llama agent ready (model:", MODEL + ")")

---
## Part 6 — Rules of thumb

| Tool type | Idempotent? | How to make it safe |
|---|---|---|
| **Read** (get, list, search) | ✅ already safe | nothing to do |
| **Calculate** | ✅ already safe | nothing to do |
| **Create** (send email, place order, charge) | ❌ dangerous | add an `idempotency_key`, dedupe before acting |
| **Update** a record | ⚠️ depends | *set* to a final value, don't *increment* |
| **Delete** | ⚠️ depends | check 'already deleted?' first, treat as success |

### Production notes

- The `processed` dict here is in-memory and resets when the kernel restarts. In production use **Redis** (with a TTL) or a **unique column in your database** so the dedup survives restarts and works across multiple workers.
- Store the **result** under the key, not just the key — so a duplicate call can return the original `message_id`.
- Prefer a **deterministic key** (hash of the meaningful inputs) when there's no natural business id.

> ✅ **Takeaway:** Structured output made the agent *predictable*. Idempotency makes it *safe to retry*. Both together = production-ready tools.